# 01 - Load in Patient Data

Prep Output Direction

In [ ]:
# Specify where you want to save your results to
out_dir = '/Users/cu135/Partners HealthCare Dropbox/Calvin Howard/studies/raynor_network_mapping/results/schmahmann_discovery_clustering_mapping/individual_symptom_specificity/TotalBarsScore'

Import Data

In [ ]:
# Specify the path to your CSV file containing NIFTI paths
input_csv_path = '/Volumes/OneTouch/01p_Schmahmann_SCA_Atrophy/results/optimzation/optimized_master_list.csv'
sheet = None

In [ ]:
from calvin_utils.permutation_analysis_utils.statsmodels_palm import CalvinStatsmodelsPalm
# Instantiate the PalmPrepararation class
cal_palm = CalvinStatsmodelsPalm(input_csv_path=input_csv_path, output_dir=out_dir, sheet=sheet)
# Call the process_nifti_paths method
data_df = cal_palm.read_and_display_data()
data_df

**Handle NANs**
- Set drop_nans=True is you would like to remove NaNs from data
- Provide a column name or a list of column names to remove NaNs from

In [ ]:
for col in data_df.columns:
    print(col)

In [ ]:
drop_list = ['TotalBarsScore', 'TotalCCASFailScore', 'AffectTotal', 'selected_Nifti_File_Path']

In [ ]:
data_df = cal_palm.drop_nans_from_columns(columns_to_drop_from=drop_list)

**Drop Row Based on Value of Column**

Define the column, condition, and value for dropping rows
- column = 'your_column_name'
- condition = 'above'  # Options: 'equal', 'above', 'below'

In [ ]:
data_df.columns

Set the parameters for dropping rows

In [ ]:
column = 'focal_cerebellum'  # The column you'd like to evaluate
condition = 'not'  # The condition to check ('equal', 'above', 'below', 'not')
value = 1 # The value to drop if found

In [ ]:
# data_df, other_df = cal_palm.drop_rows_based_on_value(column, condition, value)
display(data_df)

# 02 - Load in the Patient Maps

This uses the column in the above CSV which stored all of the file paths in it
- For example, from the base_directory, */tissue_segment_z_scores will look for all subjects, all session folders within subjects, and then check the tissue_segment_z_scores folder. 


In [ ]:
file_column = 'selected_Nifti_File_Path'

In [ ]:
from calvin_utils.file_utils.import_functions import GiiNiiFileImport
dv_df = GiiNiiFileImport(import_path=data_df[file_column], file_pattern=None, file_column=None, process_special_values=False).run()
dv_df

# 03 - Load Map to Check Damage Within

Import Region of Interest Masks

In [ ]:
base_directory = '/Users/cu135/Partners HealthCare Dropbox/Calvin Howard/studies/raynor_network_mapping/results/schmahmann_discovery_clustering_mapping/all_columns_no_perm_palm/regression_4'
shared_glob_pattern = 'contrast_tval_0.nii.gz'

In [ ]:
from calvin_utils.file_utils.import_functions import GiiNiiFileImport
iv_df = GiiNiiFileImport(import_path=base_directory, file_pattern=shared_glob_pattern).run()
iv_df

Extract Damage Scores Per Region of Interest

In [ ]:
mask_path = '/Users/cu135/Software_Local/calvin_utils_project/circuit_pyper/resources/MNI152_T1_2mm_brain_mask.nii'

In [ ]:
from calvin_utils.neuroimaging_utils.nifti_utils.damage_score_utils import DamageScorer
damage_scorer = DamageScorer(mask_path, dv_df, iv_df)
dmg_df = damage_scorer.calculate_damage_scores('cosine')
dmg_df = damage_scorer.sort_dataframes_by_index(dmg_df)
dmg_df

# 04 - Label Outcome Columns by Which Cluster they Belong To
- copy the output of the below cell and fill it out with categorical labels. 

In [ ]:
from calvin_utils.neuroimaging_utils.ccm_utils.symptom_specificity import get_column_labels
get_column_labels(data_df)

Copy tge above dictionary if you want to organize your outcomes for prediction into categories. 
- Otherwise, leave label_dict = None.

Example of a full label_dict: 
```
label_dict = {
'MotorL_acute': 'motor',
'MotorR_acute': 'motor',
'bvmt_bias_acute': 'cognitive',
'bvmt_imt_acute': 'cognitive'
}
```

In [ ]:
label_dict = {
    # Motor
    'Gait': 'Motor',
    'HeelToShinTestLeft': 'Motor',
    'HeelToShinTestRight': 'Motor',
    'FingerToNoseTestLeft': 'Motor',
    'FingerToNoseTestRight': 'Motor',
    'LimbAtaxia': 'Motor',
    'Speech': 'Motor',
    'Oculomotor': 'Motor',
    # 'TotalBarsScore': 'Motor',

    # Cognitive
    # 'SematicFluencyRawS': 'Cognitive',
    'SematicFluencyFailS': 'Cognitive',
    # 'PhonemicFluencyRawS': 'Cognitive',
    'PhonemicFluencyFailS': 'Cognitive',
    # 'CategorySwitchRawS': 'Cognitive',
    'CategorySwitchFailS': 'Cognitive',
    # 'VerbalRegSum': 'Cognitive',
    # 'DigitSpanForwardRawS': 'Cognitive',
    'DigitSpanForwardFailS': 'Cognitive',
    # 'DigitSpanBackwardRawS': 'Cognitive',
    'DigitSpanBackwardFailS': 'Cognitive',
    # 'CubeDrawRawS': 'Cognitive',
    'CubeDrawFailS': 'Cognitive',
    # 'VerbalRecallRawS': 'Cognitive',
    'VerbalRecallFailS': 'Cognitive',
    # 'SimiliarityPair1RawS': 'Cognitive',
    # 'SimiliarityPair2RawS': 'Cognitive',
    # 'SimiliarityPair3RawS': 'Cognitive',
    # 'SimiliarityPair4RawS': 'Cognitive',
    # 'SimiliarityRawS': 'Cognitive',
    'SimiliarityFailS': 'Cognitive',
    # 'GoNoGoRawS': 'Cognitive',
    'GoNoGoFailS': 'Cognitive',
    # 'TotalCCASRawScore': 'Cognitive',
    # 'TotalCCASFailScore': 'Cognitive',
    # 'MemoryTotal': 'Cognitive',
    # 'ExecutiveTotal': 'Cognitive',
    # 'LanguageTotal': 'Cognitive',
    # 'VisualTotal': 'Cognitive',

    # Emotional
    'Sec1ADifficultFocus': 'Emotional',
    'Sec1AEasilyDistracted': 'Emotional',
    'Sec1AOntheGo': 'Emotional',
    'Sec1AFeelsCompelled': 'Emotional',
    'Sec1AFeelsDriven': 'Emotional',
    'Sec1BWorries': 'Emotional',
    'Sec1BRepeats': 'Emotional',
    'Sec1BMentallyStuck': 'Emotional',
    'Sec1BCauseDistress': 'Emotional',
    'Sec3ARepetitiveMovements': 'Emotional',
    'Sec3ASensoryExp': 'Emotional',
    'Sec3BSensitive': 'Emotional',
    'Sec3BOverwhelmed': 'Emotional',
    'Sec4ACommunicates': 'Emotional',
    'Sec4AConcerns': 'Emotional',
    'Sec4ASeesHearsThings': 'Emotional',
    'Sec4BTroubleUnderstand': 'Emotional',
    'Sec4BDistant': 'Emotional',
    'Sec4BIndifferent': 'Emotional',
    # 'ScoreCol1A': 'Emotional',
    # 'ScoreCol1B': 'Emotional',
    # 'ScoreCol3A': 'Emotional',
    # 'ScoreCol3B': 'Emotional',
    # 'ScoreCol4A': 'Emotional',
    # 'ScoreCol4B': 'Emotional',
    # 'TotalSection1Score': 'Emotional',
    # 'TotalSection3Score': 'Emotional',
    # 'TotalSection4Score': 'Emotional',
    # 'CNRSTotColAScore': 'Emotional',
    # 'AFFECTAssessments_Angryoraggress': 'Emotional',
    # 'AFFECTAssessments_Difficultywith': 'Emotional',
    # 'AFFECTAssessments_Emotionallylab': 'Emotional',
    # 'AFFECTAssessments_Expressesillog': 'Emotional',
    # 'AFFECTAssessments_Lacksempathyis': 'Emotional',
    # 'AFFECTAssessments_Showseasysenso': 'Emotional',
    # 'AffectRawS': 'Emotional',
    # 'AffectFailS': 'Emotional',
    # 'AffectTotal': 'Emotional',
    'Sec2AActHastily': 'Emotional',
    'Sec2ARapidChanges': 'Emotional',
    'Sec2ACryingLaughing': 'Emotional',
    'Sec2AOverAnxious': 'Emotional',
    'Sec2BLackOfPleasure': 'Emotional',
    'Sec2BNegativeAttitude': 'Emotional',
    'Sec2BUneasyWithLife': 'Emotional',
    'Sec2BSadDepressed': 'Emotional',
    'Sec5AAngry': 'Emotional',
    'Sec5AUpset': 'Emotional',
    'Sec5AIntolerant': 'Emotional',
    'Sec5AArgumentative': 'Emotional',
    'Sec5Bimmature': 'Emotional',
    'Sec5BUnaware': 'Emotional',
    'Sec5BManner': 'Emotional',
    'Sec5BTrusting': 'Emotional',
    # 'ScoreCol2A': 'Emotional',
    # 'ScoreCol2B': 'Emotional',
    # 'ScoreCol5A': 'Emotional',
    # 'ScoreCol5B': 'Emotional',
    # 'TotalSection2Score': 'Emotional',
    # 'TotalSection5Score': 'Emotional',
#     'CNRSTotColBScore': 'Emotional',
    # 'CNRSTotScore': 'Emotional',
}

# 05 - Relate damage to symptom clusters

In [ ]:
data_df = data_df.fillna(0)
dmg_df = dmg_df.fillna(0)

In [ ]:
dmg_df

In [ ]:
from calvin_utils.neuroimaging_utils.ccm_utils.symptom_specificity import SpecificityAnalyzer

sa = SpecificityAnalyzer(X=dmg_df, Y=data_df, y_labels=label_dict, 
                    correlation='spearman', method='permutation',
                    vectorize=False, out_dir=out_dir+'/permutation', absval=False)
sa.run(n_resamples=1000, sort_within_labels=False, smooth_by_label=True)

Run Comparisons Across groups

In [ ]:
r_long = sa.extract_r_values_long()
df = sa.extract_group_comparison_r_values(
    target_labels="Cognitive",
    other_labels=None,      # None means every non-motor label is pooled
    target_name="Cognitive",
    other_name="Other",
)

display(df)

In [ ]:
from calvin_utils.plotting_utils.simple_box_plot import SimpleBoxPlotWrapper
plotter = SimpleBoxPlotWrapper(df)
plotter.plot(
    columns=[('Cognitive', 'Other')],
    pair_names=['Cognitive', 'Other'],
    dataset_name=' ',
    xlabel=' ',
    ylabel='Correlation (Spearman)',
    figsize=(4, 6),
    out_dir=out_dir+'/permutation',
)

That's all

-Calvin